<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/lab-10-operate-the-meridian-agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 10 (graded) — Operate the Meridian agent
**Course 3: AI Agents and Agentic AI with Python — Chapter 10: Deploying & operating agents**

**A month after the dispute assistant shipped — Leo Farkas:** "A tool API changed its
response format, the model provider pushed an update, one run cost $40 because it looped,
and nobody knew until the monthly bill."

**What you'll submit:** the Chapter 5 agent deployed with tracing + budget caps, contract
tests catching a simulated tool-format change and a model update, a release gate blocking a
bad graph change, and a monitoring plan + incident runbook.

## 1. A traced, budget-capped dispute agent

In [ ]:
import time

class BudgetExceeded(Exception):
    pass

class TracedDisputeAgent:
    def __init__(self, policy_tool, graph_version='v1', max_cost_usd=1.00, cost_per_step=0.02):
        self.policy_tool = policy_tool
        self.graph_version = graph_version
        self.max_cost_usd = max_cost_usd
        self.cost_per_step = cost_per_step
        self.trace_log = []

    def run(self, dispute_id, amount):
        t0 = time.perf_counter()
        cost = 0.0
        steps = []
        for step_name in ['intake', 'policy_check', 'draft', 'resolve']:
            cost += self.cost_per_step
            if cost > self.max_cost_usd:
                raise BudgetExceeded(f'Run for {dispute_id} exceeded ${self.max_cost_usd} budget cap.')
            steps.append(step_name)
            if step_name == 'policy_check':
                policy_result = self.policy_tool(amount)
        latency = time.perf_counter() - t0
        trace = {'dispute_id': dispute_id, 'amount': amount, 'steps': steps, 'policy_result': policy_result,
                  'cost_usd': round(cost, 3), 'latency_s': round(latency, 4), 'graph_version': self.graph_version}
        self.trace_log.append(trace)
        return trace

def policy_tool_v1(amount):
    """The CURRENT expected contract: returns {'action': 'escalate'|'auto_resolve'}."""
    return {'action': 'escalate' if amount >= 500 else 'auto_resolve'}

agent = TracedDisputeAgent(policy_tool_v1)
print(agent.run('DSP-100', 250))

## 2. A contract test that catches a tool-format change

In [ ]:
def contract_test_policy_tool(tool_fn):
    """Run on a schedule against the real tool — fails loudly if the shape changes, instead
    of silently breaking every dispute that flows through the graph."""
    result = tool_fn(600)
    assert isinstance(result, dict), f'expected a dict, got {type(result)}'
    assert 'action' in result, f'expected an "action" key, got keys={list(result.keys())}'
    assert result['action'] in ('escalate', 'auto_resolve'), f'unexpected action value: {result["action"]!r}'
    return True

print('Contract test on v1 (should pass):', contract_test_policy_tool(policy_tool_v1))

def policy_tool_v2_changed_format(amount):
    """Simulates the external API changing its response shape without warning."""
    return {'decision_code': 'ESC' if amount >= 500 else 'AUTO'}  # renamed key, different values!

try:
    contract_test_policy_tool(policy_tool_v2_changed_format)
    print('Contract test on v2: PASSED (unexpected!)')
except AssertionError as e:
    print(f'Contract test on v2: FAILED as expected — {e}')
    print('This is exactly the kind of silent breakage the scheduled contract test catches BEFORE production.')

## 3. A canary suite catching a model update

In [ ]:
CANARY_CASES = [(100, 'auto_resolve'), (400, 'auto_resolve'), (500, 'escalate'), (900, 'escalate'), (50, 'auto_resolve')]
# note the 400 case: it's what actually exposes a threshold shift from 500 -> 300 below,
# since every OTHER case here happens to land on the same side of both thresholds

def run_canary_suite(policy_tool):
    passed = 0
    for amount, expected in CANARY_CASES:
        result = policy_tool(amount)
        if isinstance(result, dict) and result.get('action') == expected:
            passed += 1
    return passed / len(CANARY_CASES)

print('Canary pass rate, v1:', run_canary_suite(policy_tool_v1))

def policy_tool_v1_after_model_update(amount):
    """Simulates an upstream model update subtly shifting the escalation threshold — same
    schema (so the contract test above would NOT catch this), different behavior."""
    return {'action': 'escalate' if amount >= 300 else 'auto_resolve'}  # threshold silently moved

rate_after = run_canary_suite(policy_tool_v1_after_model_update)
print('Canary pass rate, after simulated model update:', rate_after)
assert rate_after < 1.0, 'The canary suite should catch the behavior shift even though the schema is unchanged.'
print('\nCaught: same schema, different behavior — exactly what a contract test alone would miss,')
print('and exactly why the canary task suite exists as a separate layer.')

## 4. Release gate blocking a bad graph change

In [ ]:
def release_gate(policy_tool, min_canary_rate=1.0):
    rate = run_canary_suite(policy_tool)
    passed = rate >= min_canary_rate
    return passed, rate

good_passed, good_rate = release_gate(policy_tool_v1)
print(f'Release gate on the unchanged tool: {"PASSED" if good_passed else "BLOCKED"} (rate={good_rate:.0%})')

bad_passed, bad_rate = release_gate(policy_tool_v1_after_model_update)
print(f'Release gate on the drifted tool:   {"PASSED" if bad_passed else "BLOCKED"} (rate={bad_rate:.0%})')
assert not bad_passed, 'The release gate should block the drifted behavior.'

## 5. The budget cap actually stopping a runaway run

In [ ]:
cheap_agent = TracedDisputeAgent(policy_tool_v1, max_cost_usd=0.05)  # deliberately tiny budget
try:
    cheap_agent.run('DSP-RUNAWAY', 200)
    print('Did not raise — unexpected.')
except BudgetExceeded as e:
    print(f'Budget cap fired as designed: {e}')
    print('This is the control that would have caught Leo\'s $40 run in real time, not a month later.')

## 6. Monitoring plan + incident runbook (fill in)
- **What to monitor:** cost per run, contract-test pass/fail, canary pass rate, step count.
- **Alert thresholds (fill in):** _your numbers here_
- **Runbook — when the canary suite drops below 100% (fill in):** freeze new runs? Roll back
  to which version? Who's paged? When do you re-enable?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 10: Deploying & operating agents*